## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### Where `knowledge-base` goes

Put folder **`knowledge-base`** next to this notebook (folder **`13_LANGCHAIN`**):

`.../13_LANGCHAIN/knowledge-base/` with subfolders `company/`, `contracts/`, `employees/`, `products/`.

WSL path: `/home/soudas/PERSONAL/AI_LEARNING/AI_LEARNING/13_LANGCHAIN/knowledge-base/`

Run the notebook from **`13_LANGCHAIN`** (kernel **my-proj**). Part B creates **`vector_db/`** here too.

### How to read

Each code cell has a markdown cell above it: code block + table (like `colab_openai_and_hf.ipynb`).

### Pipeline in this notebook

A → load `.md` files → B → chunk → C → embed → Chroma → D → (optional) t-SNE plots.

**Next notebook:** [`rag_langchain_retriever_qa_gradio.ipynb`](./rag_langchain_retriever_qa_gradio.ipynb) — retriever + ChatOpenAI + Gradio chat (uses `vector_db/` from Part B).


### PART A: Divide our documents into chunks

### Code cell — imports

```python
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
# Local HF embeddings need ~1GB+ (torch + sentence-transformers). Not imported here.
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
```

| Line / piece | Meaning |
|--------------|---------|
| import os | Standard library — env vars, paths, basename for doc_type folders. |
| import glob | Standard library — **glob** = **Global** file pattern matching; `glob.glob()` finds paths like `knowledge-base/**/*.md`. |
| import tiktoken | Package tiktoken — count OpenAI-style tokens in text. |
| import numpy as np | Package numpy — arrays for embeddings and t-SNE. |
| from dotenv import load_dotenv | Package python-dotenv — load OPENAI_API_KEY from .env. |
| from langchain_openai import OpenAIEmbeddings | Package langchain-openai — OpenAI embedding API via LangChain. |
| from langchain_chroma import Chroma | Package langchain-chroma — persist vectors in Chroma (vector_db/). |
| from langchain_community... DirectoryLoader, TextLoader | Package langchain-community — load .md files as Document objects. |
| from langchain_text_splitters import RecursiveCharacterTextSplitter | Package langchain-text-splitters — split docs into chunks. |
| from sklearn.manifold import TSNE | Package scikit-learn — 2D/3D projection for plots (Part C). |
| import plotly.graph_objects as go | Package plotly — interactive scatter charts. |

**Install (once per env):** `pip install langchain-openai langchain-chroma langchain-community langchain-text-splitters tiktoken python-dotenv chromadb numpy scikit-learn plotly`

Run with working directory `13_LANGCHAIN` so `knowledge-base/` and `vector_db/` paths work.


In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
# Local HF embeddings need ~1GB+ (torch + sentence-transformers). Not imported here.
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/tmp/ipykernel_22274/978041534.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


### Code cell — configuration and API key

```python
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
```
#### Purpose
Set names and secrets before loading data. **MODEL** here is **not** the embedding model (that comes in Part B).

| Line | Meaning |
|------|---------|
| MODEL | Name passed to **tiktoken** so token counts match OpenAI-style tokenization. |
| db_name | Subfolder **vector_db/** under `13_LANGCHAIN` where Chroma saves vectors. |
| load_dotenv(override=True) | Read `.env` in this folder (or parent); `override=True` replaces existing env vars. |
| os.getenv('OPENAI_API_KEY') | API key for **OpenAIEmbeddings** later — never paste the full key in the notebook. |
| print first 8 chars | Sanity check only; not a security best practice for production, fine for learning. |

Put `.env` next to the notebook with: `OPENAI_API_KEY=sk-...`


In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


### Code cell — knowledge base size (characters)

```python
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")
```

#### What does `knowledge_base_path = "knowledge-base/**/*.md"` mean?

This is a **glob pattern** (not a normal file path). You pass it to `glob.glob()` so Python returns **every matching file path** as a list.

| Piece in the pattern | Meaning |
|----------------------|--------|
| `knowledge-base/` | Start under the folder named **knowledge-base** (next to this notebook). |
| `**` | “Any depth of subfolders” — employees, products, contracts, company, and anything nested below them. |
| `/` | Folder separator (works on Linux/WSL; same idea on Windows in Python). |
| `*.md` | Only files whose names end with **.md** (markdown). |

**Examples of paths that match:**

- `knowledge-base/employees/Alex Chen.md`
- `knowledge-base/products/Homellm.md`
- `knowledge-base/contracts/Contract with ....md`

**Why `recursive=True` on the next line?** In Python’s `glob`, the `**` “walk all subfolders” rule only applies when you set `recursive=True`. Without it, you might get fewer (or no) matches.

This cell does **not** load LangChain yet — it is a quick “how big is our corpus?” check in **characters** before chunking and embedding.

| Line / piece | Meaning |
|--------------|---------|
| knowledge_base_path | String pattern `"knowledge-base/**/*.md"` — all markdown files under the knowledge base tree. |
| glob.glob(..., recursive=True) | **glob** = Global file pattern matching — build a list of real file paths that match the pattern. |
| for file_path in files | Read each file and append to one big string. |
| len(entire_knowledge_base) | Total characters — sanity check before chunking. |

Run with working directory `13_LANGCHAIN` so `knowledge-base/` and `vector_db/` paths work.


In [3]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


### Code cell — knowledge base size (tokens)

```python
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")
```
#### Purpose
Same corpus as the character-count cell, but measured in **tokens** (pieces the model/API uses).

| Line | Meaning |
|------|---------|
| encoding_for_model(MODEL) | Get the tokenizer OpenAI associates with that model name. |
| encode(entire_knowledge_base) | Convert the big string to a list of integer token IDs. |
| len(tokens) | Total tokens (~63k here). Embedding APIs also charge per token. |

**Characters vs tokens:** ~4 characters per token is a rough rule of thumb in English; your two numbers will not match exactly.


In [4]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 63,555


### Code cell — load documents with LangChain loaders

```python
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")
```

#### What is `DirectoryLoader`?

A **LangChain document loader** class (from `langchain_community`). It means: **“read many files from a directory.”**

| Argument | In this notebook | Meaning |
|----------|------------------|--------|
| 1st arg (`folder`) | e.g. `knowledge-base/employees` | **Which directory** to scan. |
| `glob="**/*.md"` | all markdown under that folder | **Which files** to include (`**` = any subfolder). |
| `loader_cls=TextLoader` | use `TextLoader` per file | **How** to read each file (plain text, not PDF/HTML). |
| `loader_kwargs={'encoding': 'utf-8'}` | UTF-8 | **How** to decode bytes (needed for special characters in names/text). |

You **construct** the loader first (`loader = DirectoryLoader(...)`). Nothing is read from disk until you call **`.load()`**.

#### What is `loader.load()`?

**`.load()`** runs the loader: find matching files → read each one → return a **Python list** of **`Document`** objects.

| Each `Document` has | Example |
|---------------------|--------|
| `page_content` | Full text of the `.md` file |
| `metadata` | Dict, often includes `source` = file path on disk |

So: **`DirectoryLoader`** = recipe (“which folder, which files, which reader”). **`loader.load()`** = execute recipe and get **`folder_docs`** (list of Documents).

#### What this cell does (plain English)

Turn every `.md` file under `knowledge-base/` into LangChain **`Document`** objects and collect them in one list **`documents`**. Each file = one Document (until we chunk later).

#### Step-by-step

| Step | Code | What happens |
|------|------|----------------|
| 1 | `folders = glob.glob("knowledge-base/*")` | List **top-level folders** only, e.g. `knowledge-base/employees`, `knowledge-base/products`, … (**glob** = Global file pattern matching). |
| 2 | `documents = []` | Empty list; we will append every loaded file here. |
| 3 | `for folder in folders:` | Loop **once per category folder** (4 times if you have company, contracts, employees, products). |
| 4 | `doc_type = os.path.basename(folder)` | Take the **folder name** as a label, e.g. `employees` (not the full path). Used later for plot colors and filtering. |
| 5 | `DirectoryLoader(folder, glob="**/*.md", ...)` | LangChain: “inside this folder, find **all** `.md` files in **any** subfolder.” `TextLoader` reads each file as plain text. `encoding='utf-8'` handles names like `Alex Chen.md`. |
| 6 | `folder_docs = loader.load()` | Actually read files → list of **`Document`** objects. Each has `page_content` (text) and `metadata` (e.g. `source` = file path). |
| 7 | `for doc in folder_docs:` | Inner loop: every file in **this** category folder. |
| 8 | `doc.metadata["doc_type"] = doc_type` | Add our tag, e.g. `doc_type: "employees"`, on top of LangChain’s `source` metadata. |
| 9 | `documents.append(doc)` | Add to the **master list** for the whole knowledge base. |
| 10 | `print(len(documents))` | Should be **76** (one Document per `.md` file). |

#### Tiny example (one iteration)

When `folder = "knowledge-base/employees"`:

- `doc_type` → `"employees"`
- Loader reads `Oliver Spencer.md`, `Alex Chen.md`, …
- Each becomes a Document; metadata gets `doc_type: employees` and `source: knowledge-base/employees/....md`

#### Why not one loader on the whole tree?

We **loop per top folder** so **`doc_type`** is easy (basename). You could load `knowledge-base` in one shot, but you would need another way to set category labels.

Run with working directory `13_LANGCHAIN` so `knowledge-base/` paths work.


In [5]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


### Code cell — inspect one Document

```python
documents[1]
```
#### Purpose
**Debug / learn:** see what one loaded file looks like before chunking.

| Piece | What you see |
|-------|----------------|
| documents[1] | Second file in the list (index 1). Index 0 works too. |
| page_content | Full markdown text of that employee/product file. |
| metadata['source'] | Path to the file on disk. |
| metadata['doc_type'] | Label we added: employees, products, contracts, or company. |

One **Document** = one **whole file** at this stage (not a chunk yet).


In [6]:
documents[1]

Document(metadata={'source': 'knowledge-base/employees/Oliver Spencer.md', 'doc_type': 'employees'}, page_content='# HR Record\n\n# Oliver Spencer\n\n## Summary\n- **Date of Birth**: May 14, 1990\n- **Job Title**: Backend Software Engineer\n- **Location**: Austin, Texas\n- **Current Salary**: $125,000  \n\n## Insurellm Career Progression\n- **March 2018**: Joined Insurellm as a Backend Developer I, focusing on API development for customer management systems.\n- **July 2019**: Promoted to Backend Developer II after successfully leading a team project to revamp the claims processing system, reducing response time by 30%.\n- **June 2021**: Transitioned to Backend Software Engineer with a broader role in architecture and system design, collaborating closely with the DevOps team.\n- **September 2022**: Assigned as the lead engineer for the new "Innovate" initiative, aimed at integrating AI-driven solutions into existing products.\n- **January 2023**: Awarded a mentorship role to guide new h

### Code cell — split into chunks

```python
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")
```

#### Why chunk at all?

Each file was **one** `Document` (76 files). For RAG you retrieve **small pieces**, not whole employee HR files. Embedding and search work better on **~paragraph-sized** text.

After this cell: **76 documents → ~413 chunks** (many chunks per long file).

#### If a file is 1001 characters, is the last character lost?

**No.** `chunk_size=1000` is a **target max per chunk**, not “keep 1000 and delete the rest.”

| File size | What happens |
|-----------|----------------|
| ≤ 1000 chars | **One chunk** — entire employee file fits in a single piece. |
| 1001 chars | **Two chunks** — e.g. chunk 1 ≈ first 1000 chars, chunk 2 has the **remaining** text (plus **overlap** with chunk 1). |
| 5000 chars | **Several chunks** — all text is still covered across the list; nothing is truncated away. |

The splitter **splits**, it does not **discard**. Every character from the file should appear in at least one chunk’s `page_content`.

**Overlap (`chunk_overlap=200`)** means the end of chunk 1 and the start of chunk 2 **share** some text — so facts near a cut line still appear in full in at least one chunk (helps retrieval).

At **query time**, RAG usually returns **top-k chunks** (e.g. 3–5), not one chunk per employee — so a long HR file may need **multiple chunks** retrieved together to answer one question.

#### What is `RecursiveCharacterTextSplitter`?

A LangChain **text splitter** (`langchain_text_splitters`). It cuts long text into smaller strings while trying to split on **natural boundaries** (paragraph → line → space → character). **Recursive** = try big breaks first, then smaller if a piece is still too long.

#### Line 1: `text_splitter = RecursiveCharacterTextSplitter(...)`

Build the **splitter object** (settings only — no splitting yet).

| Parameter | Value here | Meaning |
|-----------|------------|--------|
| `chunk_size` | `1000` | Target **maximum** size of each chunk in **characters** (not tokens). A chunk is often a bit under 1000 after splitting on boundaries. |
| `chunk_overlap` | `200` | Neighbour chunks **share** 200 characters. So if a sentence sits on a cut, it still appears in full in at least one chunk. |

**Overlap picture (simplified):**

```text
[-------- chunk 1: 1000 chars --------]
                    [---- overlap 200 ----][-------- chunk 2 --------]
```

#### Line 2: `chunks = text_splitter.split_documents(documents)`

**`.split_documents(documents)`** runs the splitter on **every** Document in the list:

1. Take `page_content` from one file.
2. Split into pieces ≤ `chunk_size` (with overlap).
3. Each piece becomes a **new** `Document` (metadata like `source` and `doc_type` is **copied** from the parent file).
4. Repeat for all 76 files → one flat list **`chunks`**.

| Before | After |
|--------|-------|
| `documents` — 1 per `.md` file | `chunks` — many per file if file is long |
| 76 items | ~413 items (your run) |

#### What you print at the end

- `len(chunks)` — how many pieces will be embedded in Part B.
- `chunks[0]` — first chunk’s text + metadata (inspect size and labels).

Run with working directory `13_LANGCHAIN` so paths work.


In [7]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# HR Record

# Robert Chen

## Summary
- **Date of Birth:** February 28, 1983
- **Job Title:** Senior Full Stack Engineer
- **Location:** San Francisco, California
- **Current Salary:** $152,000

## Insurellm Career Progression
- **January 2016 - Present:** Senior Full Stack Engineer
  - Technical lead for Homellm home insurance portal
  - Architects full-stack solutions using React, Node.js, and PostgreSQL
  - Mentors team of 6 engineers
  - Led 3 major platform releases

- **June 2012 - December 2015:** Full Stack Developer at WebSolutions Inc.
  - Built web applications for enterprise clients
  - Worked with various JavaScript frameworks and backend technologies

- **August 2008 - May 2012:** Software Engineer at TechStartup
  - Developed features for SaaS platform
  - Gained experience across full technology stack' metadata={'source': 'knowledge-base/employees/Robert Chen.md', 'doc_type': 'employees'}


### Code cell — inspect one chunk

```python
chunks[100]
```
#### Purpose
Compare a **chunk** to a full **Document**: shorter text, same metadata family.

| Piece | What you see |
|-------|----------------|
| chunks[100] | The 101st chunk across the whole knowledge base (any file). |
| page_content | Only a **section** of a long markdown file (~1000 chars). |
| metadata | Still has source + doc_type from the parent file. |

If chunk 100 is from Priya Sharma.md, the **source** path still points to that file — retrieval will return this slice, not the entire HR record.


In [8]:
chunks[100]

Document(metadata={'source': 'knowledge-base/employees/Priya Sharma.md', 'doc_type': 'employees'}, page_content='## Other HR Notes\n- **Education:** PhD in Computer Science (Machine Learning) from Stanford University\n- **Publications:** 7+ peer-reviewed papers in ML conferences and journals\n- **Speaking:** Regular speaker at ML and InsurTech conferences. Represented Insurellm at 4 major conferences in 2023.\n- **Patents:** Co-inventor on 2 pending patents for insurance ML applications\n- **Recognition:** Data Science Excellence Award 2023, featured in InsureTech Innovation Magazine\n- **Skills:** Expert in Python, TensorFlow, PyTorch, scikit-learn, SQL, and cloud ML platforms\n- **Feedback:** World-class technical talent with strong business acumen. Natural leader who elevates entire team. Key retention priority.')

### PART B: Make vectors and store in Chroma

**Goal:** Each **chunk** → embedding vector → stored in **Chroma** (`vector_db/`).

Flow: `chunks` → OpenAI Embeddings API → vectors → disk.

This notebook uses **OpenAIEmbeddings** (API). Course material may use **HuggingFaceEmbeddings** (local model) instead — same idea, different provider.

You need **OPENAI_API_KEY** in `.env` from the config cell.


### Code cell — embed chunks and store in Chroma

```python
# Pick an embedding model

# Small local install: skip HF (needs torch + sentence-transformers). Uses API instead.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # pip: torch (cpu) + sentence-transformers

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")
```

#### What this cell does (one sentence)

Take all **413 chunks**, call OpenAI to turn each into a **vector**, save vectors + text in **Chroma** on disk under **vector_db/**.

#### Line-by-line (focus on the Chroma block)

| # | Code | Explanation |
|---|------|-------------|
| 1 | `embeddings = OpenAIEmbeddings(...)` | Create an **embedding client** (settings + API key). Does **not** embed yet — like creating `DirectoryLoader` before `.load()`. Model `text-embedding-3-small` → each chunk becomes **1536 floats**. |
| 2 | `if os.path.exists(db_name):` | `db_name` is `"vector_db"`. If that **folder** already exists from an earlier notebook run, we clean up before rebuilding. |
| 3 | `Chroma(persist_directory=db_name, embedding_function=embeddings)` | Open a connection to the **on-disk** Chroma database in `vector_db/`. `embedding_function` tells Chroma **how** to embed text if it needs to embed later (e.g. new queries). |
| 4 | `.delete_collection()` | **Delete** the old vector index inside that folder. Without this, you might **append** or conflict with stale data (old chunk count, old embeddings). Safe for learning; in production you might version DBs instead. |
| 5 | `vectorstore = Chroma.from_documents(...)` | **Main work:** loop over every item in `chunks`, embed each `page_content`, write **vector + text + metadata** to Chroma. Returns a LangChain **vectorstore** object. |
| 5a | `documents=chunks` | Parameter is named `documents` in the API, but we pass **chunks** (still LangChain `Document` objects). |
| 5b | `embedding=embeddings` | Use our OpenAI embedder for every chunk (API calls — costs tokens). |
| 5c | `persist_directory=db_name` | Save to `13_LANGCHAIN/vector_db/` so you do not re-embed every time you open the notebook. |
| 6 | `vectorstore._collection.count()` | Ask Chroma: how many vectors stored? Should match **len(chunks)** (413). |

#### What is Chroma?

A **vector database** (library + files on disk). For each chunk it stores:

- the **text** (and metadata like `source`, `doc_type`)
- the **embedding** (list of numbers)

Later, for a **user question**, you embed the question the same way and Chroma finds **nearest** chunk vectors (similarity search) — that is RAG retrieval.

#### Flow diagram

```text
chunks[0..412]  →  OpenAIEmbeddings (API)  →  413 vectors
       ↓
  Chroma writes to vector_db/
       ↓
  vectorstore  (use .as_retriever() in a later notebook)
```

#### Common confusion

| Question | Answer |
|----------|--------|
| Why delete first? | Fresh rebuild after you change chunks or re-run Part A. |
| Is data sent to OpenAI? | **Yes** — each chunk’s text is sent to the embeddings API. |
| Folder location? | `.../13_LANGCHAIN/vector_db/` next to this notebook. |


In [9]:
# Pick an embedding model

# Small local install: skip HF (needs torch + sentence-transformers). Uses API instead.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # pip: torch (cpu) + sentence-transformers

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 413 documents


### Code cell — vector dimensions and count

```python
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
```

#### What this cell does (plain English)

After Part B, Chroma on disk holds **one embedding per chunk**. This cell is a **health check**: “How many vectors did we store, and how long is each vector?”

It does **not** search or answer questions — only **inspect** what was saved.

#### Words you need

| Word | Meaning |
|------|--------|
| **vector** / **embedding** | A list of numbers (e.g. 1536 floats) that represents the **meaning** of a chunk of text. Similar text → similar numbers. |
| **vectorstore** | LangChain object from the previous cell — wrapper around Chroma. |
| **collection** | Chroma’s internal table of all stored rows (text + vector + metadata). |
| **dimensions** | How many numbers in **one** vector (1536 for `text-embedding-3-small`). |

#### Line-by-line

| Line | Code | What it does |
|------|------|----------------|
| 1 | `collection = vectorstore._collection` | `vectorstore` is LangChain’s high-level API. `._collection` reaches **inside** Chroma to the raw collection object so we can call `.count()` and `.get()`. (The leading `_` means “internal” — fine for learning notebooks.) |
| 2 | `count = collection.count()` | Ask Chroma: **how many rows** are in the database? Should match **len(chunks)** (e.g. **413**). If you see 0, Part B did not run or failed. |
| 3 | `collection.get(limit=1, include=["embeddings"])` | **Fetch** data from Chroma. `limit=1` = only **one** row. `include=["embeddings"]` = also return the **numeric vector**, not just text. Returns a **dict** with keys like `embeddings`, `documents`, `metadatas`. |
| 4 | `["embeddings"][0]` | From that dict: take the list of embeddings, then the **first** (and only) vector. Result: a Python list of ~1536 floats, e.g. `[0.012, -0.034, ...]`. |
| 5 | `dimensions = len(sample_embedding)` | Count how many numbers in that list → **1536**. Every chunk uses the same length for this model. |
| 6 | `print(...)` | Human-readable summary: **413 vectors**, each **1536-dimensional**. |

#### Tiny picture

```text
Chunk text:  "Robert Chen, Senior Engineer..."
       ↓  (Part B already did this)
Embedding:   [0.02, -0.11, 0.08, ... ]  ← 1536 numbers
       ↓  (this cell)
count() → 413 such rows in Chroma
get()   → peek at one row’s numbers
```

#### Why care?

- **count** = did all chunks get embedded?
- **dimensions** = must stay the same when you later embed **user questions** for search (same model).

Part C will call `collection.get(...)` again to pull **all** embeddings for the plot.


In [10]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 1,536 dimensions in the vector store


### Part C: Visualize!

#### What Part C is for (and what it is NOT)

| | Part B | Part C |
|---|--------|--------|
| **Does** | Save 413 embeddings in Chroma | **Look** at those embeddings as a picture |
| **For RAG production?** | **Yes** — retrieval uses these vectors | **No** — plots are for **learning / intuition** only |
| **User question answered?** | Not yet | Still **no** — no LLM, no retriever here |

You already built the **search index** in Part B. Part C asks: “Do chunks from the same folder (employees vs products) land **near each other** in embedding space?”

#### The problem: 1536 dimensions

Each chunk is a point in **1536-dimensional** space. You cannot draw that on a screen.

**t-SNE** (t-distributed Stochastic Neighbor Embedding) is a math trick that squeezes each 1536-D point down to **2 numbers (x, y)** or **3 numbers (x, y, z)** so Plotly can scatter-plot them.

**Important:** t-SNE distances are **not** what Chroma uses for search. Chroma uses full 1536-D similarity. The plot is a **rough map**, not the real retrieval engine.

#### Part C has 3 code cells

1. **Prework** — load all vectors + labels + colors from Chroma  
2. **2D plot** — t-SNE → scatter chart  
3. **3D plot** — same idea with a z axis (rotate in the UI)

#### Color legend (same in both plots)

| doc_type | Color |
|----------|-------|
| products | blue |
| employees | green |
| contracts | red |
| company | orange |


### Code cell — prepare data for visualization

```python
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
```
#### What this cell does (plain English)

Export **everything** from Chroma into Python variables so the next cells can plot. You need **`collection`** from the previous cell (`vectorstore._collection`).

#### Line-by-line

| Line | Code | Explanation |
|------|------|-------------|
| 1 | `result = collection.get(include=[...])` | **Read ALL rows** from Chroma (not `limit=1` like the inspect cell). Ask for three parallel lists: embeddings, chunk text, metadata. |
| 2 | `vectors = np.array(result['embeddings'])` | Turn list of 413 embeddings into a **NumPy array** shape **(413, 1536)** — row = one chunk, column = one dimension. |
| 3 | `documents = result['documents']` | List of **413 strings** — the text of each chunk. **Warning:** variable name `documents` here shadows Part A’s `documents` list; in Part C it means **chunk texts**. |
| 4 | `metadatas = result['metadatas']` | List of 413 dicts (`source`, `doc_type`, …). |
| 5 | `doc_types = [metadata['doc_type'] for ...]` | One label per chunk: `employees`, `products`, `contracts`, or `company`. |
| 6 | `colors = [...]` | **List comprehension:** for each `doc_type`, pick a color. `index(t)` maps type → 0..3 → blue/green/red/orange. Same length as chunks — **one color per dot** in the plot. |

#### What `result` looks like (conceptually)

```text
result = {
  'embeddings': [ [1536 floats], [1536 floats], ... ],  # 413 items
  'documents':  [ "chunk text...", ... ],
  'metadatas': [ {'source': '...', 'doc_type': 'employees'}, ... ]
}
```

#### After this cell

Variables ready for t-SNE: **`vectors`**, **`doc_types`**, **`colors`**, and **`documents`** (for hover text).


In [11]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

### Code cell — 2D t-SNE plot

```python
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()
```
#### What this cell does (plain English)

1. Run **t-SNE** to turn each 413×1536 point into one **(x, y)** pair.  
2. Draw a **scatter plot** — each dot = one chunk; color = folder type; hover = type + text preview.

#### Step 1 — t-SNE

| Line | Code | Explanation |
|------|------|-------------|
| | `TSNE(n_components=2, random_state=42)` | Build a t-SNE object: output **2D**. `random_state=42` → same layout if you re-run (mostly). |
| | `reduced_vectors = tsne.fit_transform(vectors)` | **Heavy step** — can take ~30s–2min. Input `(413, 1536)` → output `(413, 2)`. |

**Analogy:** 413 cities with 1536 coordinates each → t-SNE places them on a **flat map** keeping nearby cities somewhat close.

#### Step 2 — Plotly scatter

| Line | Code | Explanation |
|------|------|-------------|
| | `go.Figure(data=[go.Scatter(...)])` | Create an interactive figure. |
| | `x=reduced_vectors[:, 0]` | All **x** coordinates (column 0). |
| | `y=reduced_vectors[:, 1]` | All **y** coordinates (column 1). |
| | `mode='markers'` | Dots only, no lines. |
| | `marker=dict(size=5, color=colors, ...)` | Dot size; **color** from prework cell (one per chunk). |
| | `text=[f"Type: {t}<br>Text: {d[:100]}..."]` | **Hover** label: doc_type + first 100 chars of chunk. |
| | `hoverinfo='text'` | Show that hover text. |
| | `fig.update_layout(...)` | Title, size, margins. |
| | `fig.show()` | Render chart in the notebook. |

#### How to read the chart

- **Green blobs** → employee chunks near other employee chunks (embedding similarity).  
- **Mixed colors** → types overlap in meaning or t-SNE distorted distances.  
- **Does not mean** retrieval is wrong — it is only a 2D **guess** at structure.

#### Part B vs Part C (again)

| Retrieval (RAG) | This plot |
|-------------------|-----------|
| Full 1536-D vectors in Chroma | 2-D summary |
| Used to answer questions | Used to **see** your data |


In [12]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

### Code cell — 3D t-SNE plot

```python
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()
```
#### What this cell does

**Same pipeline as 2D**, but t-SNE outputs **3 coordinates** per chunk and Plotly draws **Scatter3d** so you can rotate/zoom.

| Line | Explanation |
|------|-------------|
| `n_components=3` | Each chunk → (x, y, z). |
| `fit_transform(vectors)` | Runs t-SNE **again** from scratch — layout will **not** match the 2D plot point-for-point. |
| `go.Scatter3d` | 3D markers; same `colors` and hover as 2D. |
| `scene=dict(xaxis_title=...)` | Axis labels for the 3D box. |

Use the chart controls to **drag** and rotate. Helpful when 2D squashes too much information.

**Remember:** Still not used for RAG — only exploration.


In [13]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()